## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. It also installs `scikit-learn`, which this analysis needs, into the same session. You can read every line of the code below before you run it.

# Three delivery apps: forward stepwise selection

**Discriminant Analysis** · a worked example from the StatNex teaching guide

**The story.** 150 food-delivery orders, 50 from each of three apps: ZipEats (cheap, close by), FeastNow (middle) and UrbanBite (premium restaurants, further away).

**The question.** Looking only at an order's numbers, can you tell which app it came from?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

This analysis needs `scikit-learn`. This cell installs them. If Colab asks you to restart the session, restart it and choose **Run all** again.

In [ ]:
%pip install -q "scikit-learn"

## The table

This cell saves the table `09_multivariate/discriminant_analysis__3groups.csv` (150 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
No.,App,Delivery time (min),Distance (km),Order total ($),Tip ($)
1.0,ZipEats,16.0,1.5,18.41,2.07
2.0,ZipEats,24.0,1.8,22.84,2.68
3.0,ZipEats,28.0,2.4,12.55,1.77
4.0,ZipEats,19.0,1.0,14.25,0.0
5.0,ZipEats,29.0,1.9,14.64,0.21
6.0,ZipEats,26.0,1.9,16.77,1.34
7.0,ZipEats,22.0,1.2,17.94,2.51
8.0,ZipEats,30.0,2.9,9.43,0.0
9.0,ZipEats,22.0,1.3,12.85,2.08
10.0,ZipEats,26.0,2.1,19.0,1.79
11.0,ZipEats,28.0,1.2,21.85,2.97
12.0,ZipEats,17.0,1.6,9.97,0.0
13.0,ZipEats,29.0,2.5,21.83,3.46
14.0,ZipEats,26.0,2.3,14.55,0.89
15.0,ZipEats,23.0,1.3,17.27,1.06
16.0,ZipEats,23.0,1.5,21.22,1.76
17.0,ZipEats,23.0,2.2,15.01,0.95
18.0,ZipEats,22.0,1.8,17.59,2.45
19.0,ZipEats,26.0,1.4,14.39,0.86
20.0,ZipEats,12.0,0.7,18.94,0.64
21.0,ZipEats,26.0,2.5,17.0,2.43
22.0,ZipEats,14.0,2.0,9.05,1.5
23.0,ZipEats,29.0,3.2,13.19,0.0
24.0,ZipEats,20.0,2.3,11.02,0.99
25.0,ZipEats,23.0,2.1,20.25,0.63
26.0,ZipEats,20.0,1.3,16.26,1.03
27.0,ZipEats,27.0,3.2,18.54,1.01
28.0,ZipEats,23.0,2.2,14.43,2.45
29.0,ZipEats,28.0,1.8,13.04,0.67
30.0,ZipEats,19.0,2.7,9.3,0.76
31.0,ZipEats,27.0,2.3,14.7,0.02
32.0,ZipEats,27.0,2.5,11.43,0.98
33.0,ZipEats,29.0,3.1,11.19,0.55
34.0,ZipEats,24.0,1.8,19.65,0.81
35.0,ZipEats,26.0,1.6,18.99,2.62
36.0,ZipEats,18.0,1.1,20.07,0.83
37.0,ZipEats,25.0,2.2,12.95,2.23
38.0,ZipEats,58.0,2.0,15.34,1.43
39.0,ZipEats,17.0,1.0,10.77,0.0
40.0,ZipEats,31.0,2.6,20.83,0.22
41.0,ZipEats,27.0,2.2,13.66,1.75
42.0,ZipEats,18.0,1.3,12.59,0.42
43.0,ZipEats,24.0,1.4,11.63,0.55
44.0,ZipEats,29.0,1.6,12.33,1.37
45.0,ZipEats,25.0,3.1,12.61,0.74
46.0,ZipEats,24.0,1.2,14.06,1.97
47.0,ZipEats,25.0,2.3,19.86,1.34
48.0,ZipEats,23.0,2.6,12.98,2.01
49.0,ZipEats,20.0,0.7,14.33,0.7
50.0,ZipEats,26.0,1.6,9.85,1.46
51.0,FeastNow,32.0,3.9,27.47,3.59
52.0,FeastNow,26.0,3.0,29.69,2.59
53.0,FeastNow,31.0,3.2,32.42,5.02
54.0,FeastNow,33.0,3.8,32.92,3.85
55.0,FeastNow,25.0,2.1,20.56,2.82
56.0,FeastNow,37.0,4.5,21.81,3.04
57.0,FeastNow,30.0,2.7,20.8,1.18
58.0,FeastNow,28.0,2.4,24.03,2.99
59.0,FeastNow,29.0,2.5,24.28,3.67
60.0,FeastNow,35.0,4.3,28.65,3.35
61.0,FeastNow,33.0,3.9,34.57,3.4
62.0,FeastNow,27.0,2.0,27.35,3.67
63.0,FeastNow,34.0,4.0,28.64,3.25
64.0,FeastNow,26.0,3.0,23.07,2.47
65.0,FeastNow,23.0,1.7,22.11,2.36
66.0,FeastNow,30.0,2.8,26.63,3.97
67.0,FeastNow,30.0,3.2,28.15,1.64
68.0,FeastNow,33.0,2.6,31.1,3.49
69.0,FeastNow,35.0,4.4,26.51,2.67
70.0,FeastNow,29.0,2.1,26.86,2.88
71.0,FeastNow,29.0,3.2,14.45,1.71
72.0,FeastNow,42.0,5.2,26.27,2.15
73.0,FeastNow,33.0,3.5,27.28,3.67
74.0,FeastNow,32.0,2.7,28.93,2.74
75.0,FeastNow,32.0,3.3,32.57,3.64
76.0,FeastNow,41.0,4.0,18.17,0.39
77.0,FeastNow,29.0,3.6,28.68,1.87
78.0,FeastNow,36.0,3.6,34.65,2.93
79.0,FeastNow,34.0,3.3,38.95,3.52
80.0,FeastNow,33.0,2.8,25.84,2.02
81.0,FeastNow,30.0,4.3,23.9,3.4
82.0,FeastNow,32.0,3.0,20.64,2.89
83.0,FeastNow,38.0,3.9,26.52,2.64
84.0,FeastNow,35.0,3.4,23.54,2.48
85.0,FeastNow,41.0,5.1,32.35,3.14
86.0,FeastNow,31.0,4.2,19.05,1.34
87.0,FeastNow,37.0,4.6,22.4,2.18
88.0,FeastNow,37.0,3.9,18.2,3.07
89.0,FeastNow,33.0,4.2,25.6,3.21
90.0,FeastNow,31.0,3.3,21.64,0.92
91.0,FeastNow,40.0,5.4,29.35,3.64
92.0,FeastNow,39.0,4.6,21.83,2.31
93.0,FeastNow,35.0,3.6,24.4,2.99
94.0,FeastNow,37.0,3.7,12.35,1.42
95.0,FeastNow,35.0,3.8,27.95,2.41
96.0,FeastNow,35.0,4.0,22.95,1.44
97.0,FeastNow,31.0,3.9,31.9,4.72
98.0,FeastNow,33.0,3.7,28.75,4.27
99.0,FeastNow,37.0,4.6,32.45,3.61
100.0,FeastNow,33.0,3.1,27.47,2.34
101.0,UrbanBite,36.0,3.4,36.32,4.2
102.0,UrbanBite,41.0,4.7,29.45,4.63
103.0,UrbanBite,38.0,5.4,34.75,4.73
104.0,UrbanBite,48.0,7.1,54.77,8.42
105.0,UrbanBite,33.0,3.3,53.88,7.15
106.0,UrbanBite,42.0,6.3,31.78,4.1
107.0,UrbanBite,42.0,5.7,53.32,6.23
108.0,UrbanBite,42.0,4.4,22.94,3.46
109.0,UrbanBite,34.0,4.0,42.1,4.72
110.0,UrbanBite,46.0,6.0,56.02,7.98
111.0,UrbanBite,38.0,4.0,32.99,3.89
112.0,UrbanBite,53.0,7.3,49.04,8.06
113.0,UrbanBite,41.0,4.7,37.75,5.02
114.0,UrbanBite,45.0,5.1,38.8,6.29
115.0,UrbanBite,35.0,5.0,27.69,3.33
116.0,UrbanBite,46.0,6.2,61.51,8.77
117.0,UrbanBite,47.0,6.6,36.88,5.75
118.0,UrbanBite,33.0,4.2,38.32,5.06
119.0,UrbanBite,44.0,5.3,50.62,8.78
120.0,UrbanBite,31.0,2.9,68.51,9.35
121.0,UrbanBite,32.0,3.6,36.39,4.71
122.0,UrbanBite,44.0,6.9,38.89,6.6
123.0,UrbanBite,42.0,5.4,41.79,6.62
124.0,UrbanBite,39.0,3.9,46.31,6.35
125.0,UrbanBite,41.0,4.4,48.27,6.46
126.0,UrbanBite,38.0,5.2,56.79,9.08
127.0,UrbanBite,46.0,6.3,49.88,5.92
128.0,UrbanBite,38.0,4.6,44.09,6.71
129.0,UrbanBite,36.0,4.6,56.16,7.64
130.0,UrbanBite,31.0,4.3,32.96,4.45
131.0,UrbanBite,36.0,5.3,45.07,6.59
132.0,UrbanBite,38.0,4.5,57.9,7.52
133.0,UrbanBite,32.0,3.2,37.21,5.18
134.0,UrbanBite,46.0,5.8,48.05,7.86
135.0,UrbanBite,45.0,5.4,48.31,7.0
136.0,UrbanBite,38.0,4.5,39.96,5.37
137.0,UrbanBite,44.0,5.1,42.82,6.5
138.0,UrbanBite,34.0,3.7,56.6,8.21
139.0,UrbanBite,32.0,3.0,42.9,5.07
140.0,UrbanBite,29.0,3.1,35.49,5.55
141.0,UrbanBite,45.0,5.6,54.84,5.94
142.0,UrbanBite,40.0,4.7,39.08,4.98
143.0,UrbanBite,37.0,4.1,44.59,7.24
144.0,UrbanBite,42.0,5.4,44.91,5.67
145.0,UrbanBite,43.0,5.2,49.36,6.07
146.0,UrbanBite,36.0,3.4,46.85,6.8
147.0,UrbanBite,39.0,4.1,51.69,7.05
148.0,UrbanBite,40.0,5.7,44.44,4.64
149.0,UrbanBite,46.0,4.7,51.58,8.62
150.0,UrbanBite,44.0,5.7,47.37,6.35
""")

## Helper: collinear_columns

A step that no library does, written out. In the code download it is the file `collinear_columns.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Which explanatory variables are exact linear combinations of the ones before them.

Such a variable adds no information and makes the model impossible to estimate. Each
column is centred and scaled to length 1, then regressed on the columns already kept;
if nothing is left over (squared residual length at most 1e-10) the column is
redundant. A constant column is redundant too. The *later* member of a redundant pair
is the one reported, so a variable listed first survives.
"""

import numpy as np


def collinear_columns(values, names, tolerance=1e-10):
    centred = values - values.mean(axis=0)
    scale = np.linalg.norm(centred, axis=0)
    scale[scale == 0] = 1.0
    centred = centred / scale
    dropped, kept = [], []
    for j, name in enumerate(names):
        column = centred[:, j]
        if kept:
            basis = centred[:, kept]
            column = column - basis @ np.linalg.lstsq(basis, column, rcond=None)[0]
        if column @ column <= tolerance or values[:, j].max() == values[:, j].min():
            dropped.append(name)
        else:
            kept.append(j)
    return dropped

## Helper: discriminant

A step that no library does, written out. In the code download it is the file `discriminant.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""The arithmetic of canonical discriminant analysis that SciPy and statsmodels lack.

W is the within-group sums of squares and cross-products, B the between-group one.
Wilks' lambda = |W| / |W + B| is the share of the variables' variation the groups do
*not* explain; everything below is built from it.
"""

import numpy as np
from scipy import stats


def scatter_matrices(values, codes, groups):
    """W (within groups), B (between groups) and the grand mean."""
    grand = values.mean(axis=0)
    p = values.shape[1]
    within, between = np.zeros((p, p)), np.zeros((p, p))
    for k in range(groups):
        block = values[codes == k]
        centred = block - block.mean(axis=0)
        within += centred.T @ centred
        shift = (block.mean(axis=0) - grand)[:, None]
        between += len(block) * shift @ shift.T
    return within, between, grand


def wilks(values, codes, groups):
    """Wilks' lambda of the given variables (1 when there are none)."""
    if values.shape[1] == 0:
        return 1.0
    within, between, _ = scatter_matrices(values, codes, groups)
    return np.exp(np.linalg.slogdet(within)[1] - np.linalg.slogdet(within + between)[1])


def rao_f(wilks_lambda, p, g, n):
    """Rao's F approximation to Wilks' lambda: F, df1, df2, P (exact when min(p, g-1) <= 2)."""
    df1 = p * (g - 1)
    numerator, denominator = p**2 * (g - 1) ** 2 - 4, p**2 + (g - 1) ** 2 - 5
    s = np.sqrt(numerator / denominator) if numerator > 0 and denominator > 0 else 1.0
    df2 = (n - 1 - (p + g) / 2) * s - (df1 - 2) / 2
    root = wilks_lambda ** (1 / s)
    f = (1 - root) / root * df2 / df1
    return f, df1, df2, stats.f.sf(f, df1, df2)


def partial_f(full, reduced, g, n, variables):
    """Partial lambda (full / reduced) of one variable given the others, its F, df and P.

    `variables` counts the model that includes the variable.
    """
    partial = full / reduced
    df1, df2 = g - 1, n - g - variables + 1
    if df2 <= 0:
        return partial, np.nan, df1, max(df2, 0), np.nan
    f = (1 - partial) / partial * df2 / df1
    return partial, f, df1, df2, stats.f.sf(f, df1, df2)


def box_m(samples):
    """Box's M test of equal covariance matrices, in its chi-square form: M, chi2, df, P."""
    usable = [s for s in samples if s.shape[0] > s.shape[1]]
    p, k = usable[0].shape[1], len(usable)
    dfs = np.array([len(s) - 1 for s in usable], dtype=float)
    covariances = [np.cov(s, rowvar=False) for s in usable]
    pooled = sum(d * c for d, c in zip(dfs, covariances)) / dfs.sum()
    m = dfs.sum() * np.linalg.slogdet(pooled)[1] - sum(
        d * np.linalg.slogdet(c)[1] for d, c in zip(dfs, covariances))
    c1 = ((1 / dfs).sum() - 1 / dfs.sum()) * (2 * p * p + 3 * p - 1) / (6 * (p + 1) * (k - 1))
    df = (k - 1) * p * (p + 1) // 2
    return m, m * (1 - c1), df, stats.chi2.sf(m * (1 - c1), df)


def stepwise(values, codes, g, start, may_enter, may_remove, use_p, enter, remove):
    """Variable selection on partial Wilks' lambda; one record per step.

    Each step enters the best variable that qualifies (smallest P, or largest F); only
    when none does, it removes the worst that qualifies for removal (largest P, or
    smallest F). It stops when a step makes no move or would revisit a set.
    """
    n = len(codes)
    pool = range(values.shape[1])
    current, steps, seen = list(start), [], set()
    while tuple(current) not in seen:
        seen.add(tuple(current))
        base = wilks(values[:, current], codes, g)
        entered = removed = None
        if may_enter:
            best = None
            for j in pool:
                if j in current:
                    continue
                full = wilks(values[:, [*current, j]], codes, g)
                _, f, _, df2, p = partial_f(full, base, g, n, len(current) + 1)
                if df2 <= 0 or not np.isfinite(f):
                    continue
                qualifies, score = (p < enter, p) if use_p else (f > enter, -f)
                if qualifies and (best is None or score < best[0]):
                    best = (score, j)
            entered = best[1] if best else None
        if may_remove and entered is None and current:
            worst = None
            for j in current:
                reduced = wilks(values[:, [c for c in current if c != j]], codes, g)
                _, f, _, df2, p = partial_f(base, reduced, g, n, len(current))
                if df2 <= 0 or not np.isfinite(f):
                    continue
                qualifies, score = (p > remove, -p) if use_p else (f < remove, f)
                if qualifies and (worst is None or score < worst[0]):
                    worst = (score, j)
            removed = worst[1] if worst else None
        steps.append({"before": list(current), "entered": entered, "removed": removed})
        if entered is None and removed is None:
            break
        if entered is not None:
            current.append(entered)
        if removed is not None:
            current.remove(removed)
    return steps, current

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import cdist
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
# collinear_columns: defined in the helper cell collinear_columns above (collinear_columns.py in the code download)
# box_m, partial_f, rao_f, scatter_matrices and wilks: defined in the helper cell discriminant above (discriminant.py in the code download)
# stepwise: defined in the helper cell discriminant above (discriminant.py in the code download)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## The data

In [ ]:
# The response names each case's group; text in an explanatory column counts as missing.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
response = "App"
predictors = ["Delivery time (min)", "Distance (km)", "Order total ($)", "Tip ($)"]
frame = data[predictors].apply(pd.to_numeric, errors="coerce")


# A group as text; a number read as 2.0 prints as "2".
def label(value):
    return str(int(value)) if isinstance(value, float) and value.is_integer() else str(value)

## Cases

In [ ]:
y_ok, x_ok = data[response].notna(), frame.notna().all(axis=1)
summary = pd.DataFrame(
    {"n": [(y_ok & x_ok).sum(), (~y_ok & x_ok).sum(), (y_ok & ~x_ok).sum(),
           (~y_ok & ~x_ok).sum(), len(frame)]},
    index=["valid", "response missing", "explanatory missing", "both missing", "total"],
)
summary["%"] = summary["n"] / len(frame)
print(summary)
valid = frame[y_ok & x_ok]
outcome = data.loc[y_ok & x_ok, response].map(label).to_numpy()
# Groups in the order they first appear, for now (they are reordered after the fit).
groups = list(dict.fromkeys(outcome))
codes = np.array([groups.index(v) for v in outcome])
values = valid.to_numpy(float)
n, g = len(codes), len(groups)

## Variables that are linear combinations of others

In [ ]:
dropped = collinear_columns(values, predictors)
print("\nLinearly dependent:", dropped or "none")
all_predictors = list(predictors)
# They are left out of the analysis.
predictors = [name for name in predictors if name not in dropped]
values = valid[predictors].to_numpy(float)

## Variable selection on Wilks' lambda

In [ ]:
# A variable's contribution is its partial lambda -- lambda with it over lambda without
# it -- tested by F on (g - 1, n - g - k + 1) degrees of freedom.
# Forward stepwise: start empty; enter the best qualifying variable, or, when none
# qualifies, remove the worst one that qualifies for removal.
start, may_enter, may_remove = [], True, True
# Criterion: P value. Enter below the entry value, remove above the removal value.
use_p = True
enter, remove = 0.2, 0.2
selection_settings = [enter, remove]
steps, chosen = stepwise(values, codes, g, start, may_enter, may_remove, use_p, enter, remove)
names = [predictors[j] for j in chosen]
print("Variables in the functions:", names)

## The discriminant functions

In [ ]:
# scikit-learn's LinearDiscriminantAnalysis finds the weights that maximise
# between-group over within-group variation (the eigenvectors of W⁻¹B), at most
# min(p, g - 1) of them, each scaled so its scores have a pooled within-group variance
# of 1. They are signed here so their weights add up to a positive number: a function's
# sign is arbitrary, and this fixes it as StatNex does. transform() gives the scores,
# each case's value on each function measured from the overall means.
X = values[:, chosen]
within, between, grand = scatter_matrices(X, codes, g)
pooled = within / (n - g)
lda = LinearDiscriminantAnalysis(solver="svd").fit(X, codes)
# Convention: scikit-learn scales the scores so their within-group variance divides by
# n; MASS::lda, SPSS and StatNex divide by n - g, as a pooled variance does. Each
# function is rescaled to that (on 150 cases in 3 groups, by sqrt(147 / 150)).
spread = np.sqrt(np.diag(lda.scalings_.T @ pooled @ lda.scalings_))
signs = np.where(lda.scalings_.sum(axis=0) < 0, -1.0, 1.0)
vectors = lda.scalings_ / spread * signs
scores = lda.transform(X) / spread * signs
functions = [f"function {j + 1}" for j in range(vectors.shape[1])]
# Each function's eigenvalue is its between-group sum of squares over its within-group
# sum of squares: how many times the separation of the groups outweighs the scatter
# inside them. scikit-learn keeps only their shares, so they are worked out from the scores.
group_means = np.array([scores[codes == k].mean(axis=0) for k in range(g)])
between_ss = (np.bincount(codes, minlength=g)[:, None] * group_means**2).sum(axis=0)
within_ss = ((scores - group_means[codes]) ** 2).sum(axis=0)
eigenvalues = between_ss / within_ss

# The groups are listed by their centroid on the first function, largest first.
centroids = np.array([scores[codes == k].mean(axis=0) for k in range(g)])
order = list(np.argsort(-centroids[:, 0]))
groups = [groups[k] for k in order]
codes = np.array([order.index(c) for c in codes])
centroids = pd.DataFrame(centroids[order], index=groups, columns=functions)

## Groups, descriptive statistics and correlations (every variable given)

In [ ]:
response_summary = pd.DataFrame({"n": [*np.bincount(codes, minlength=g), n]}, index=[*groups, "total"])
response_summary["%"] = response_summary["n"] / n
parts = {"all": valid, **{name: valid[codes == k] for k, name in enumerate(groups)}}
descriptive = pd.concat({
    name: pd.DataFrame({"n": len(part), "mean": part[all_predictors].mean(),
                        "variance": part[all_predictors].var(ddof=1),
                        "SD": part[all_predictors].std(ddof=1), "min": part[all_predictors].min(),
                        "max": part[all_predictors].max()})
    for name, part in parts.items()
})
correlation = pd.concat({name: part[all_predictors].corr() for name, part in parts.items()})
print()
print(response_summary)
print()
print(descriptive)
print()
print(correlation)

## Box's M test of equal covariance matrices

In [ ]:
# On every variable left after the collinearity check, in its chi-square form. It is
# very sensitive on large samples: read it as a prompt to look, not as a verdict.
box = list(box_m([values[codes == k] for k in range(g)]))
print("Box's M, chi-square, df, P:", box)

## The selection, step by step

In [ ]:
# Each step shows the variables in before its move, the variables in and out with
# their partial lambda, and which variable the move enters or removes.
blank = [np.nan] * 5
process, step_included, step_excluded = [], [], []
for number, step in enumerate(steps):
    current = step["before"]
    base = wilks(values[:, current], codes, g)
    if current:
        process.append([len(current), 1 - base, base, *rao_f(base, len(current), g, n)])
    else:
        process.append([0, *[np.nan] * 6])
    inside = [list(partial_f(base, wilks(values[:, [c for c in current if c != j]], codes, g),
                             g, n, len(current))) for j in current]
    outside = [list(partial_f(wilks(values[:, [*current, j]], codes, g), base, g, n, len(current) + 1))
               for j in range(len(predictors)) if j not in current]
    step_included += [[number, *row] for row in inside or [blank]]
    step_excluded += [[number, *row] for row in outside or [blank]]
columns = ["lambda", "F", "df1", "df2", "P"]
process = pd.DataFrame(process, columns=["variables", "eta2", "lambda", "F", "df1", "df2", "P"])
process["entered"] = [predictors[s["entered"]] if s["entered"] is not None else None for s in steps]
process["removed"] = [predictors[s["removed"]] if s["removed"] is not None else None for s in steps]
step_included = pd.DataFrame(step_included, columns=["step", *columns])
step_excluded = pd.DataFrame(step_excluded, columns=["step", *columns])
print()
print(process)
print()
print(step_included)
print()
print(step_excluded)

## Significance of the selected variables

In [ ]:
# Wilks' lambda of the whole set, by Rao's F; then each variable's partial lambda.
k = len(chosen)
lam = wilks(X, codes, g)
selection_result = [k, 1 - lam, lam, *rao_f(lam, k, g, n)]
included = pd.DataFrame(
    [partial_f(lam, wilks(np.delete(X, j, axis=1), codes, g), g, n, k) for j in range(k)],
    index=names, columns=["lambda", "F", "df1", "df2", "P"])
print("\nvariables, eta2, lambda, F, df1, df2, P:", selection_result)
print(included)

# Eigenvalues: each function's share of the separation, and its correlation ratio.
eigen = pd.DataFrame({"function": np.arange(1, len(eigenvalues) + 1), "eigenvalue": eigenvalues,
                      "share": eigenvalues / eigenvalues.sum(),
                      "cumulative": np.cumsum(eigenvalues) / eigenvalues.sum(),
                      "eta2": eigenvalues / (1 + eigenvalues)})
# Bartlett's sequential test: is anything left from function j onwards?
function_tests = []
for j in range(len(eigenvalues)):
    remaining = np.prod(1 / (1 + eigenvalues[j:]))
    chi2 = -(n - 1 - (k + g) / 2) * np.log(remaining)
    df = (k - j) * (g - j - 1)
    function_tests.append([remaining, chi2, df, stats.chi2.sf(chi2, df)])
function_tests = pd.DataFrame(function_tests, columns=["lambda", "chi-square", "df", "P"])
print()
print(eigen.to_string(index=False))
print()
print(function_tests)

# Raw weights with the constant that centres the scores; standardised weights are the
# raw ones times each variable's pooled within-group SD.
coefficients = pd.DataFrame(np.vstack([vectors, -grand @ vectors]), index=[*names, "Constant"],
                            columns=functions)
standardised = pd.DataFrame(vectors * np.sqrt(np.diag(pooled))[:, None], index=names,
                            columns=functions)
print()
print(coefficients)
print()
print(standardised)

## Classification

In [ ]:
# Each case goes to the group whose mean is nearest in Mahalanobis distance (pooled
# within-group covariance): the discriminant rule once every group is given the same
# prior probability, so a large group does not attract the borderline cases. The
# squared distances are printed with the scores.
means = np.array([X[codes == k].mean(axis=0) for k in range(g)])
distances = cdist(X, means, metric="mahalanobis", VI=np.linalg.inv(pooled)) ** 2
equal_priors = LinearDiscriminantAnalysis(solver="svd", priors=np.full(g, 1 / g)).fit(X, codes)
predicted = equal_priors.predict(X)
classification = pd.crosstab(codes, predicted).reindex(index=range(g), columns=range(g), fill_value=0)
classification.index = classification.columns = groups
classification["hit rate"] = np.diag(classification) / classification.sum(axis=1)
hit_rate = (predicted == codes).mean()
print()
print(classification)
print(f"overall hit rate: {hit_rate:.4f}")
print()
print(centroids)

## Discriminant scores and squared distances to each group

In [ ]:
case_scores = pd.DataFrame(np.column_stack([scores, distances]), columns=[*functions, *groups])
case_scores.insert(0, "predicted", [groups[k] for k in predicted])
case_scores.insert(0, "observed", [groups[k] for k in codes])
print()
print(case_scores.to_string())

## What the result says

Discriminant analysis builds scores that separate known groups, here orders from three apps. Forward stepwise selection adds one predictor at a time while it still helps. It took order total first, then distance, then tip. Delivery time stayed out: with the others in, its P value, the chance of a gain this large if it added nothing, is 0.55. So no further help from it is shown.

Wilks' lambda, the share of the variation that the groups do not explain, falls to 0.14, with a P value of almost zero. The first discriminant function carries 98% of the separation. The classification result puts 90% of the 150 orders with the right app: 49 of 50 for ZipEats, 45 for FeastNow and 41 for UrbanBite. Every mistake is between neighbours on the price ladder. Box's M test (P almost zero) says the three apps do not have the same spread, so read the hit rates with some care.

So mostly, yes: order size, distance and tip tell the apps apart.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.